# CMS Medicare Part D — Lenalidomide Geographic Analysis

**Prerequisite:** Run `00_build_database.ipynb` first.

Lenalidomide (Revlimid) is an oral immunomodulatory drug used primarily for multiple myeloma. It was the highest-cost drug in Medicare Part D in 2023 at $3.86 billion nationally. This module explores geographic variation in prescribing across U.S. states.

In [1]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

db_path = r"C:\Users\palla\OneDrive\Documents\Coding Projects\CMS\database\cms.db"
conn = sqlite3.connect(db_path)

## Step 1: National Summary

Confirm national totals and identify all lenalidomide name variants in the dataset.

In [2]:
national = pd.read_sql_query("""
    SELECT Brnd_Name, Gnrc_Name, Tot_Prscrbrs, Tot_Clms, Tot_Benes,
           ROUND(Tot_Drug_Cst, 0) AS Tot_Drug_Cst,
           ROUND(Tot_Drug_Cst / Tot_Benes, 0) AS Cost_Per_Bene
    FROM part_d_geo
    WHERE Prscrbr_Geo_Lvl = 'National'
      AND LOWER(Gnrc_Name) LIKE '%lenalidomide%'
    ORDER BY Tot_Clms DESC
""", conn)

national

,Brnd_Name,Gnrc_Name,Tot_Prscrbrs,Tot_Clms,Tot_Benes,Tot_Drug_Cst,Cost_Per_Bene
0,Revlimid,Lenalidomide,11315,216143,36944.0,3.858656e+09,104446.0
1,Lenalidomide,Lenalidomide,8741,121722,20385.0,1.680454e+09,82436.0


## Step 2: State Coverage Check

How many states have lenalidomide data? Full 50-state coverage is needed for a choropleth map.

In [3]:
states = pd.read_sql_query("""
    SELECT Prscrbr_Geo_Desc, Prscrbr_Geo_Cd,
           SUM(Tot_Prscrbrs) AS Tot_Prscrbrs,
           SUM(Tot_Clms) AS Tot_Clms,
           SUM(Tot_Benes) AS Tot_Benes,
           ROUND(SUM(Tot_Drug_Cst), 0) AS Tot_Drug_Cst
    FROM part_d_geo
    WHERE Prscrbr_Geo_Lvl = 'State'
      AND LOWER(Gnrc_Name) LIKE '%lenalidomide%'
    GROUP BY Prscrbr_Geo_Desc, Prscrbr_Geo_Cd
    ORDER BY Tot_Clms DESC
""", conn)

print(f"States with lenalidomide data: {len(states)}")
states

States with lenalidomide data: 55


,Prscrbr_Geo_Desc,Prscrbr_Geo_Cd,Tot_Prscrbrs,Tot_Clms,Tot_Benes,Tot_Drug_Cst
0,California,06,2150,36275,6241.0,559009380.0
1,Florida,12,1352,26891,4633.0,447113863.0
2,New York,36,1667,25414,4863.0,416899834.0
3,Texas,48,1323,21596,4316.0,351125306.0
4,Pennsylvania,42,1035,14080,2788.0,226556790.0
5,Georgia,13,589,12910,2041.0,218351697.0
6,Ohio,39,733,12782,2377.0,210607467.0
7,North Carolina,37,639,12649,2232.0,210623627.0
8,Michigan,26,584,12076,2027.0,203489126.0
9,Illinois,17,709,12072,2088.0,211201897.0
